In [ ]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import numpy as np
import random
import polars
import pyarrow
import datetime
import statistics
import pandas as pd
import polars as pl

from IPython.core.magic import register_cell_magic
import time
@register_cell_magic
def cpu_time(line, cell):
    try:
        n = int(line.strip()) if line.strip() else 5
    except ValueError:
        print("Uso: %%cpu_time [n_iteraciones]")
        return

    tiempos = []

    for _ in range(n):
        start = time.process_time()
        exec(cell, globals())
        end = time.process_time()
        tiempos.append(end - start)

    media = statistics.mean(tiempos)
    desviacion = statistics.stdev(tiempos) if n > 1 else 0.0

    print(f"Ejecutado {n} veces")
    print(f"Tiempo medio de CPU: {media:.6f} s")
    print(f"Desviación típica:   {desviacion:.6f} s")

random.seed(64)

def generate_probabilities_normal(n,mu,sigma):

    arr = np.zeros(n)

    for i in range(len(arr)):
        x = i - mu
        arr[i] = math.exp(-(x**2)/(2*sigma**2)) / (sigma*math.sqrt(2*math.pi))

    arr = arr / arr.sum()

    return(arr)

def generate_probabilities_exp(n,lamda):

    arr = np.zeros(n)

    for i in range(len(arr)):
        arr[i] = lamda * math.exp(-lamda * i)

    arr = arr / arr.sum()

    return(arr)


def binary_encoding(word_list):

    num_bits = math.ceil(math.log2(len(word_list)))


    binary_dict = {}
    for i, word in enumerate(word_list):
        binary_dict[word] = format(i, f"0{num_bits}b")


    binary_array = np.array([int(binary_dict[word], 2) for word in word_list])
    binary_list = binary_array.tolist()

    return binary_list


In [ ]:
countries = pd.read_excel('Population.xlsx').sort_values(by='Population').head(100)
total = int(countries['Population'].sum())
countries['Prob'] = countries['Population']/total
countries = countries.sort_values(by=['Prob'],ascending=False)
countries = countries.reset_index()
p_country = countries['Prob']

badges = pd.read_excel('occupation.xlsx',sheet_name='Table 1.1',header=1)
badges = badges.rename(columns={'2021 National Employment Matrix title': 'badges', 'Employment, 2021' : 'employees'})
badges = badges.drop(index=0)[['badges','employees']].dropna().head(4)

gender = ['M','F']
bday = [i for i in range(365)]
age_range = [1,2,3,4]

total = badges['employees'].sum()
badges['prob'] = badges['employees']/total
badges = badges.sort_values(by=['prob'],ascending=False)
badges = badges.reset_index()
p_badges = badges['prob']
p_badges = [0.4,0.34,0.16,0.1]

print(p_badges)

badges = np.array(binary_encoding(badges.index))
country = np.array(binary_encoding(countries.index))
age_range = np.array(binary_encoding(age_range))

age_range = list(np.arange(1,5))
currency = list(np.arange(8))

p_age_range = [0.2,0.3,0.4,0.1]
p_currency = generate_probabilities_normal(len(currency),len(currency)/2,len(currency)/2)
p_gender = [0.20,0.8]
p_bday = [1/365]*365

df_app2 = pd.DataFrame(columns=['country','bday','badges','gender','currency','age'])

N = 10_000_000

df_app2 = pd.DataFrame({
	'country': np.random.choice(country, N, p=p_country),
	'bday': np.random.choice(bday, N),
    'badges': np.random.choice(badges, N, p=p_badges),
    'gender': np.random.choice(gender, N),
    'currency': np.random.choice(currency, N, p=p_currency),
    'age_range': np.random.choice(age_range, N, p=p_age_range),
    },dtype='category')

In [ ]:
countries

In [ ]:
df_app2

In [ ]:
attr = ['country','age_range','badges','currency','bday']

In [ ]:
%%cpu_time 100
comb_count = (
    df_app2.groupby(attr,observed=True)
    .size()
    .reset_index(name='count')
    .sort_values('count')
    .reset_index()
)

In [ ]:
%%cpu_time 100
comb_count.loc[0]['count']

In [ ]:
%%cpu_time 100
comb_count['cumsum'] = comb_count['count'].cumsum()
suma = comb_count['count'].sum()
comb_count['cdf'] = comb_count['cumsum'] / suma

res_geq099 = comb_count[comb_count['cdf'] >= 0.01]
idx_min = (res_geq099['cdf'] - 0.01).abs().idxmin()
results_99 = res_geq099.loc[idx_min]
results_99